# Step 4b: Köppen Panel Models — DT, RF & Wilcoxon Test
### DI 501 Term Project | Eda Yilmaz

**Goal:**  
Train tuned Decision Tree and Random Forest models on the three Köppen-Geiger panels
(Mediterranean, Temperate, Continental) using 9 European countries. Apply Wilcoxon
signed-rank tests to compare model performance against the naive baseline.
Results feed into Step 8b for direct comparison with data-driven cluster results.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_predict, StratifiedKFold, KFold
from scipy.stats import wilcoxon

plt.rcParams["figure.dpi"] = 120

# ── Load European panel dataset ───────────────────────────────────────────────
df_ml  = pd.read_csv("./master_ml_dataset.csv")    # has actual yield + panel
df_sel = pd.read_csv("./master_ml_selected.csv")   # has heat_stress_days, prec_gs, detrended_yield, panel

FEATURES = ["heat_stress_days", "prec_gs"]
TARGET   = "detrended_yield"
PANELS   = ["Mediterranean", "Temperate", "Continental"]

print("Dataset loaded.")
print(f"  European countries: {sorted(df_ml['country'].unique())}")
print(f"  Panels: {df_ml['panel'].unique().tolist()}")
print(f"  Shape: {df_sel.shape}")
print()

# Mean actual yield per panel (for rRMSE denominator)
mean_yield_per_panel = df_ml.groupby("panel")["yield"].mean()
print("Mean actual yield per panel (rRMSE denominator):")
print(mean_yield_per_panel.to_string())

Dataset loaded.
  European countries: ['AT', 'DE', 'EL', 'ES', 'FR', 'HU', 'IT', 'PL', 'RO']
  Panels: ['Temperate', 'Mediterranean', 'Continental']
  Shape: (10662, 6)

Mean actual yield per panel (rRMSE denominator):
panel
Continental      4.989454
Mediterranean    8.119730
Temperate        8.990384


---
## 1. Tune RF and Get Cross-Validated Predictions per Panel

In [2]:
from sklearn.tree import DecisionTreeRegressor

param_grid_rf = {
    "rf__n_estimators":     [100, 200],
    "rf__max_depth":        [5, 10, None],
    "rf__min_samples_leaf": [5, 10],
}
param_grid_dt = {
    "dt__max_depth":        [3, 5, 10, None],
    "dt__min_samples_leaf": [5, 10, 20],
}

results = {}

for panel in PANELS:
    pdf = df_sel[df_sel["panel"] == panel].dropna(subset=FEATURES + [TARGET]).copy()
    X = pdf[FEATURES].values
    y = pdf[TARGET].values
    mean_y = mean_yield_per_panel[panel]
    y_naive = np.zeros_like(y)

    # ── Tune RF ───────────────────────────────────────────────────────────────
    pipe_rf = Pipeline([("scaler", StandardScaler()), ("rf", RandomForestRegressor(random_state=42, n_jobs=-1))])
    gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
    gs_rf.fit(X, y)
    y_pred_rf = cross_val_predict(gs_rf.best_estimator_, X, y, cv=5)

    # ── Tune DT ───────────────────────────────────────────────────────────────
    pipe_dt = Pipeline([("scaler", StandardScaler()), ("dt", DecisionTreeRegressor(random_state=42))])
    gs_dt = GridSearchCV(pipe_dt, param_grid_dt, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
    gs_dt.fit(X, y)
    y_pred_dt = cross_val_predict(gs_dt.best_estimator_, X, y, cv=5)

    def rrmse(y_true, y_pred, my):
        return (np.sqrt(((y_true - y_pred)**2).mean()) / my) * 100

    abs_err_rf    = np.abs(y - y_pred_rf)
    abs_err_dt    = np.abs(y - y_pred_dt)
    abs_err_naive = np.abs(y - y_naive)

    stat_rf, p_rf = wilcoxon(abs_err_rf, abs_err_naive, alternative="less")
    stat_dt, p_dt = wilcoxon(abs_err_dt, abs_err_naive, alternative="less")

    results[panel] = {
        "n":           len(y),
        "rrmse_naive": rrmse(y, y_naive, mean_y),
        "rrmse_dt":    rrmse(y, y_pred_dt, mean_y),
        "rrmse_rf":    rrmse(y, y_pred_rf, mean_y),
        "w_rf": stat_rf, "p_rf": p_rf, "sig_rf": p_rf < 0.05,
        "w_dt": stat_dt, "p_dt": p_dt, "sig_dt": p_dt < 0.05,
        "best_rf": gs_rf.best_params_,
        "best_dt": gs_dt.best_params_,
    }
    print(f"{panel}: naive={results[panel]['rrmse_naive']:.1f}%, DT={results[panel]['rrmse_dt']:.1f}% (p={p_dt:.4f}), RF={results[panel]['rrmse_rf']:.1f}% (p={p_rf:.4f})")

print("Done.")

Mediterranean: naive=17.2%, DT=17.3% (p=1.0000), RF=17.4% (p=1.0000)
Temperate: naive=13.4%, DT=12.2% (p=0.0000), RF=12.2% (p=0.0000)
Continental: naive=24.0%, DT=23.0% (p=0.0049), RF=22.9% (p=0.0047)
Done.


---
## 2. Wilcoxon Results Table

In [3]:
print("WILCOXON SIGNED-RANK TEST: Tuned Models vs Naive Baseline — Koppen Panels")
print("H0: No difference | H1: Model < Naive (one-sided)")
print("Significance level: alpha = 0.05")
print()
col_header = f"{'Panel':<16} {'N':>6} {'Naive':>8} {'DT':>8} {'DT p':>8} {'RF':>8} {'RF p':>8}  {'DT sig':<14} {'RF sig'}"
print(col_header)
print("-" * 90)
for panel in PANELS:
    r = results[panel]
    dt_sig = "SIGNIFICANT *" if r["sig_dt"] else "not sig."
    rf_sig = "SIGNIFICANT *" if r["sig_rf"] else "not sig."
    print(f"{panel:<16} {r['n']:>6} {r['rrmse_naive']:>7.1f}% {r['rrmse_dt']:>7.1f}% {r['p_dt']:>8.4f} {r['rrmse_rf']:>7.1f}% {r['p_rf']:>8.4f}  {dt_sig:<14} {rf_sig}")
print()
print("COMPARISON SUMMARY:")
sig_rf = sum(1 for r in results.values() if r["sig_rf"])
sig_dt = sum(1 for r in results.values() if r["sig_dt"])
print(f"  Koppen panels — RF: {sig_rf}/3 significant | DT: {sig_dt}/3 significant  (9 European countries)")
print(f"  Data-driven clusters — RF: 2/3 significant | DT: see Step 8            (36 countries)")

WILCOXON SIGNED-RANK TEST: Tuned Models vs Naive Baseline — Koppen Panels
H0: No difference | H1: Model < Naive (one-sided)
Significance level: alpha = 0.05

Panel                 N    Naive       DT     DT p       RF     RF p  DT sig         RF sig
------------------------------------------------------------------------------------------
Mediterranean      3304    17.2%    17.3%   1.0000    17.4%   1.0000  not sig.       not sig.
Temperate          5851    13.4%    12.2%   0.0000    12.2%   0.0000  SIGNIFICANT *  SIGNIFICANT *
Continental        1507    24.0%    23.0%   0.0049    22.9%   0.0047  SIGNIFICANT *  SIGNIFICANT *

COMPARISON SUMMARY:
  Koppen panels — RF: 2/3 significant | DT: 2/3 significant  (9 European countries)
  Data-driven clusters — RF: 2/3 significant | DT: see Step 8            (36 countries)
